# DSR Dataset Generation (Colab-ready)

This notebook is intended for the dissertation project's MANET/DSR data-generation pipeline.

Important: do not start the full 180-scenario batch from this notebook unless you explicitly want to run the simulations. The environment and manifest validation sections are safe to execute first.

In [2]:
import os
import platform
import shutil
import sys
from pathlib import Path

def detect_environment():
    info = {
        'platform': platform.system(),
        'release': platform.release(),
        'python_version': platform.python_version(),
        'is_colab': 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG')),
        'cwd': str(Path.cwd()),
        'env_vars': {
            'HOME': os.environ.get('HOME'),
            'PWD': os.environ.get('PWD'),
            'USER': os.environ.get('USER'),
            'COLAB_RELEASE_TAG': os.environ.get('COLAB_RELEASE_TAG')
        },
    }
    return info

env_info = detect_environment()
print('Environment summary:')
for key, value in env_info.items():
    if isinstance(value, dict):
        print(f'  {key}:')
        for subkey, subvalue in value.items():
            print(f'    {subkey} = {subvalue}')
    else:
        print(f'  {key} = {value}')

Environment summary:
  platform = Linux
  release = 6.6.122+
  python_version = 3.13.15
  is_colab = True
  cwd = /content
  env_vars:
    HOME = /root
    PWD = /
    USER = None
    COLAB_RELEASE_TAG = release-colab-external-images_20260930-060043_RC00


In [3]:
def detect_gpu():
    candidates = [
        '/usr/bin/nvidia-smi',
        '/usr/local/cuda/bin/nvidia-smi',
    ]
    for path in candidates:
        if Path(path).exists():
            return {'available': True, 'binary': path}
    return {'available': False, 'binary': None}

def get_system_resources():
    try:
        import psutil
        memory = psutil.virtual_memory()
        cpu_count = psutil.cpu_count()
        disk = shutil.disk_usage('/')
        return {
            'cpu_count': cpu_count,
            'memory_gb': round(memory.total / (1024 ** 3), 2),
            'disk_total_gb': round(disk.total / (1024 ** 3), 2),
            'disk_free_gb': round(disk.free / (1024 ** 3), 2),
        }
    except Exception:
        return {'cpu_count': os.cpu_count(), 'memory_gb': None, 'disk_total_gb': None, 'disk_free_gb': None}

gpu = detect_gpu()
resources = get_system_resources()
print('GPU detection:')
print(gpu)
print('System resources:')
print(resources)

candidates = [
    Path.cwd(),
    Path('/content'),
    Path('/workspace'),
    Path('/Users/bistupaul/Movies/DISSERTATION'),
]
project_root = None
for base in candidates:
    if base.exists():
        for candidate in [base, base / 'DISSERTATION', base / 'workspace', base / 'project']:
            if candidate.exists() and (candidate / 'data').exists() and (candidate / 'ml').exists():
                project_root = candidate
                break
        if project_root is not None:
            break
print(f'Project root candidate: {project_root}')
if project_root is not None:
    print('Project root structure:')
    for child in sorted((project_root / 'data').iterdir())[:10]:
        print(' ', child.name)
    print('  ...')

GPU detection:
{'available': False, 'binary': None}
System resources:
{'cpu_count': 2, 'memory_gb': 12.67, 'disk_total_gb': 107.72, 'disk_free_gb': 87.23}
Project root candidate: None


In [4]:
ns3_candidates = [
    Path('/content/ns-3.47'),
    Path('/workspace/ns-3.47'),
    Path('/usr/local/src/ns-3.47'),
    Path.cwd() / 'ns3',
    Path('/Users/bistupaul/Movies/DISSERTATION/ns3'),
]

ns3_dir = None
for candidate in ns3_candidates:
    if candidate.exists():
        has_binary = (candidate / 'ns3').exists() or (candidate / 'scratch').exists()
        if has_binary or any(candidate.iterdir()):
            ns3_dir = candidate
            break

print('ns-3 candidate directories:')
for candidate in ns3_candidates:
    print(f'  {candidate}: exists={candidate.exists()}')
print(f'Chosen ns-3 directory: {ns3_dir}')
if ns3_dir is not None:
    print('ns-3 root files:')
    for child in sorted(ns3_dir.iterdir())[:10]:
        print(f'  {child.name}')


ns-3 candidate directories:
  /content/ns-3.47: exists=False
  /workspace/ns-3.47: exists=False
  /usr/local/src/ns-3.47: exists=False
  /content/ns3: exists=False
  /Users/bistupaul/Movies/DISSERTATION/ns3: exists=False
Chosen ns-3 directory: None


## Manifest validation

This next cell validates the master metadata manifest before running any scenario. It checks that the CSV exists and contains the expected DSR fields.

No simulation runs are triggered here.

In [5]:
import pandas as pd

metadata_path = None
if project_root is not None:
    metadata_path = project_root / 'data' / 'metadata' / 'Simulation_data.csv'
if metadata_path is None:
    metadata_path = Path('/content') / 'DISSERTATION' / 'data' / 'metadata' / 'Simulation_data.csv'
if metadata_path is None:
    metadata_path = Path.cwd() / 'data' / 'metadata' / 'Simulation_data.csv'

print(f'Manifest path: {metadata_path}')
print(f'Exists: {metadata_path.exists()}')

required_columns = [
    'scenario_id', 'run_id', 'random_seed', 'random_run', 'simulation_duration_s',
    'node_count', 'area_m', 'area_x_m', 'area_y_m', 'mobility_model',
    'mobility_level', 'speed_mps', 'source_id', 'destination_id',
    'traffic_level', 'traffic_rate_bps', 'packet_rate_pps', 'packet_size_bytes',
    'transport_protocol', 'application_type', 'routing_protocol', 'channel_helper',
    'wifi_standard', 'phy', 'tx_power_dbm', 'noise_floor_dbm', 'flow_id',
    'seed_source', 'endpoint_source', 'configuration_source', 'metadata_source',
    'configuration_version'
]

if metadata_path.exists():
    df = pd.read_csv(metadata_path)
    missing = [col for col in required_columns if col not in df.columns]
    print(f'Row count: {len(df)}')
    print(f"Unique scenario IDs: {df['scenario_id'].nunique() if 'scenario_id' in df.columns else 0}")
    print(f'Missing columns: {missing}')
    print('Preview columns:', list(df.columns[:12]))
    print(df.head(3).to_string(index=False))
    if missing:
        print('Manifest is not ready for any simulation batch.')
    else:
        print('Manifest structure looks valid; still verify topology and mobility coverage before running scenarios.')
else:
    print('Manifest file is missing. Create or regenerate it before any scenario execution.')


Manifest path: /content/DISSERTATION/data/metadata/Simulation_data.csv
Exists: False
Manifest file is missing. Create or regenerate it before any scenario execution.


In [5]:
if project_root is not None:
    manifest = project_root / 'data' / 'metadata' / 'Simulation_data.csv'
    topology = project_root / 'data' / 'topology' / 'Topology_data.csv'
    mobility = project_root / 'data' / 'mobility' / 'Mobility_data.csv'
    print('Coverage check:')
    for path in [manifest, topology, mobility]:
        print(f'  {path.name}: exists={path.exists()}')
    if manifest.exists():
        manifest_df = pd.read_csv(manifest)
        print(f"  manifest scenarios: {manifest_df['scenario_id'].nunique()}")
    if topology.exists():
        topo_df = pd.read_csv(topology)
        print(f"  topology scenarios: {topo_df['scenario_id'].nunique() if 'scenario_id' in topo_df.columns else 0}")
    if mobility.exists():
        mob_df = pd.read_csv(mobility)
        print(f"  mobility scenarios: {mob_df['scenario_id'].nunique() if 'scenario_id' in mob_df.columns else 0}")
else:
    print('Skipping coverage summary because project root was not detected.')


Skipping coverage summary because project root was not detected.


---

This notebook intentionally stops after environment and manifest checks. The simulation batch, dataset generation, and checkpoint logic are defined below but are not executed until you explicitly authorize them.

In [6]:
# Placeholder for future: scenario selection, per-scenario simulation, resume, and validation.
# This cell is intentionally not run by default.
# The project is still gated on topology/mobility coverage and DSR route-state completeness.
print('Simulation section not executed. Environment check complete.')

Simulation section not executed. Environment check complete.
